In [2]:

from pathlib import Path
# Path is used to work with file and directory paths

import pandas as pd
# Pandas is used to read the metadata.csv file

import torch
# PyTorch: tensors, model training, saving the model

from torch import nn
# nn contains neural-network layers such as Linear, ReLU, Dropout

from torch.utils.data import Dataset
# Dataset provides the structure for loading training examples

from torch.utils.data import DataLoader
# DataLoader creates batches of training data

from speaker_encoder import SpeakerEncoder
# Our WavLM-based encoder
# Converts each audio file into a numerical embedding


# ============================================================
# CONFIGURATION
# ============================================================

METADATA_FILE = (
    Path("../data/processed/timit")
    / "metadata.csv"
)
# Metadata created during TIMIT preprocessing
#
# Contains information such as:
# - audio_path
# - speaker_id
# - transcript
# - duration
# - sample_rate


MODEL_DIR = Path(
    "../models/voice_cloning"
)
# Directory where the trained speaker classifier will be saved


MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)
# Create the model directory if it does not already exist


BATCH_SIZE = 16
# Number of audio embeddings processed together
# during one training step


EPOCHS = 10
# Number of complete passes through the training dataset


LEARNING_RATE = 0.001
# Controls how quickly the neural network updates
# its weights during training


# ============================================================
# DATASET
# ============================================================

class SpeakerDataset(Dataset):
    # Custom PyTorch Dataset
    #
    # Its job is to:
    #
    # 1. Find an audio file
    # 2. Generate its WavLM embedding
    # 3. Find the corresponding speaker label
    # 4. Return:
    #
    #    embedding + speaker label

    def __init__(
        self,
        dataframe,
        speaker_to_label
    ):

        self.df = dataframe.reset_index(
            drop=True
        )
        # Store the metadata DataFrame
        #
        # reset_index() gives the DataFrame
        # a clean 0, 1, 2, 3... index


        self.speaker_to_label = (
            speaker_to_label
        )
        # Dictionary that converts speaker IDs
        # into numerical labels
        #
        # Example:
        #
        # FCJF0 → 0
        # MABC1 → 1
        # FXYZ2 → 2


        self.encoder = SpeakerEncoder()
        # Load the pretrained WavLM speaker/audio encoder
        #
        # This will convert each WAV file into
        # a fixed-size numerical embedding


    def __len__(self):

        return len(self.df)
        # Tell PyTorch how many samples are in the dataset


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]
        # Get one row from metadata.csv


        audio_path = row[
            "audio_path"
        ]
        # Get the path to the audio file


        speaker_id = row[
            "speaker_id"
        ]
        # Get the speaker ID associated with the audio


        embedding = self.encoder.encode(
            audio_path
        )
        # Send the audio through WavLM
        #
        # WAV
        #   ↓
        # WavLM
        #   ↓
        # numerical embedding


        label = self.speaker_to_label[
            speaker_id
        ]
        # Convert the speaker ID into an integer label
        #
        # Example:
        #
        # speaker_id = "FCJF0"
        #
        # label = 0


        return (
            embedding.squeeze(),
            torch.tensor(label)
        )
        # Return:
        #
        # embedding → input to classifier
        #
        # label → correct speaker class


# ============================================================
# CLASSIFIER
# ============================================================

class SpeakerClassifier(
    nn.Module
):
    # Neural-network classifier
    #
    # Input:
    #   WavLM embedding
    #
    # Output:
    #   probability scores/logits for each speaker

    def __init__(
        self,
        embedding_size,
        num_speakers
    ):

        super().__init__()
        # Initialize the parent PyTorch nn.Module


        self.network = nn.Sequential(
        # Sequential means the data passes through
        # each layer in order


            nn.Linear(
                embedding_size,
                256
            ),
            # First fully connected layer
            #
            # Input:
            #   WavLM embedding
            #
            # Output:
            #   256 features


            nn.ReLU(),
            # Non-linear activation function
            #
            # Helps the neural network learn
            # more complex relationships


            nn.Dropout(0.3),
            # Randomly turns off 30% of neurons
            # during training
            #
            # Helps reduce overfitting


            nn.Linear(
                256,
                128
            ),
            # Second fully connected layer
            #
            # 256 features → 128 features


            nn.ReLU(),
            # Another non-linear activation


            nn.Linear(
                128,
                num_speakers
            )
            # Final layer
            #
            # Produces one output for every speaker
            #
            # Example:
            #
            # 630 speakers
            # → 630 output values
        )


    def forward(self, x):

        return self.network(x)
        # Pass the embedding through the neural network
        # and return the speaker predictions


# ============================================================
# TRAIN
# ============================================================

def train():

    df = pd.read_csv(
        METADATA_FILE
    )
    # Load the TIMIT metadata


    print(
        f"Training samples: {len(df)}"
    )
    # Display the number of audio samples


    speakers = sorted(
        df["speaker_id"].unique()
    )
    # Get every unique speaker ID
    #
    # Example:
    #
    # ["FCJF0", "FDRW0", "MABC1", ...]


    speaker_to_label = {
        speaker: i
        for i, speaker in enumerate(
            speakers
        )
    }
    # Convert speaker IDs into numerical labels
    #
    # Example:
    #
    # {
    #     "FCJF0": 0,
    #     "FDRW0": 1,
    #     "MABC1": 2
    # }


    print(
        f"Number of speakers: "
        f"{len(speakers)}"
    )
    # Display the number of speaker classes


    dataset = SpeakerDataset(
        df,
        speaker_to_label
    )
    # Create the PyTorch Dataset


    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    # Create batches of training data
    #
    # batch_size=16:
    #   Process 16 embeddings at a time
    #
    # shuffle=True:
    #   Randomize the order of training examples


    # ========================================================
    # DETERMINE EMBEDDING SIZE
    # ========================================================

    sample_embedding, _ = dataset[0]
    # Get one example embedding
    #
    # "_" ignores the speaker label


    embedding_size = (
        sample_embedding.shape[-1]
    )
    # Determine the number of dimensions
    # in the WavLM embedding
    #
    # For WavLM Base Plus this is typically:
    #
    # 768 dimensions


    # ========================================================
    # CREATE CLASSIFIER
    # ========================================================

    model = SpeakerClassifier(
        embedding_size,
        len(speakers)
    )
    # Create the neural network
    #
    # Input:
    #   embedding_size
    #
    # Output:
    #   number of speakers


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )
    # Adam adjusts the neural-network weights
    # during training


    # ========================================================
    # LOSS FUNCTION
    # ========================================================

    criterion = nn.CrossEntropyLoss()
    # CrossEntropyLoss measures how far the model's
    # predicted speaker is from the correct speaker
    #
    # Lower loss = better predictions


    # ========================================================
    # TRAINING
    # ========================================================

    for epoch in range(EPOCHS):

        model.train()
        # Put the classifier into training mode
        #
        # This activates training-specific behavior
        # such as Dropout


        total_loss = 0
        # Used to accumulate the loss for the epoch


        for embeddings, labels in dataloader:

            # ------------------------------------------------
            # CLEAR OLD GRADIENTS
            # ------------------------------------------------

            optimizer.zero_grad()
            # Remove gradients from the previous training step


            # ------------------------------------------------
            # FORWARD PASS
            # ------------------------------------------------

            outputs = model(
                embeddings
            )
            # Send the WavLM embeddings through
            # the speaker classifier
            #
            # Outputs contain one score per speaker


            # ------------------------------------------------
            # CALCULATE LOSS
            # ------------------------------------------------

            loss = criterion(
                outputs,
                labels
            )
            # Compare:
            #
            # Model prediction
            #       vs.
            # Correct speaker label


            # ------------------------------------------------
            # BACKPROPAGATION
            # ------------------------------------------------

            loss.backward()
            # Calculate gradients showing how each
            # model parameter contributed to the error


            # ------------------------------------------------
            # UPDATE MODEL
            # ------------------------------------------------

            optimizer.step()
            # Update the neural-network weights
            # using the calculated gradients


            total_loss += (
                loss.item()
            )
            # Add this batch's loss to the epoch total


        # ====================================================
        # AVERAGE LOSS
        # ====================================================

        average_loss = (
            total_loss /
            len(dataloader)
        )
        # Calculate the average loss across all batches


        print(
            f"Epoch "
            f"{epoch + 1}/{EPOCHS} "
            f"- Loss: "
            f"{average_loss:.4f}"
        )
        # Display training progress


    # ========================================================
    # SAVE MODEL
    # ========================================================

    model_path = (
        MODEL_DIR /
        "speaker_classifier.pt"
    )
    # Path where the trained model will be saved


    torch.save(
        {
            "model_state_dict":
                model.state_dict(),
            # Save the learned neural-network weights


            "speaker_to_label":
                speaker_to_label,
            # Save the mapping between speaker IDs
            # and numerical labels


            "embedding_size":
                embedding_size
            # Save the embedding size so we know
            # what input dimension the classifier expects

        },
        model_path
    )
    # Save everything into one PyTorch checkpoint


    print(
        f"Model saved to "
        f"{model_path}"
    )
    # Confirm that the model was successfully saved


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    train()
    # Start the entire speaker-classification training process



/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training samples: 5670
Number of speakers: 567
Loading pretrained speaker encoder...
Speaker encoder loaded successfully.
Epoch 1/10 - Loss: 6.3480
Epoch 2/10 - Loss: 6.3448
Epoch 3/10 - Loss: 6.3448
Epoch 4/10 - Loss: 6.3441
Epoch 5/10 - Loss: 6.3429
Epoch 6/10 - Loss: 6.1419
Epoch 7/10 - Loss: 5.7793
Epoch 8/10 - Loss: 5.6287
Epoch 9/10 - Loss: 5.5545
Epoch 10/10 - Loss: 5.5024
Model saved to ../models/voice_cloning/speaker_classifier.pt
